In [1]:
from manim import *
import numpy as np
config.media_width = "75%"
config.verbosity = "WARNING"

# colors

In [32]:
%%manim -qm BumpyPlane

class BumpyPlane(ThreeDScene):
    def construct(self):
        self.set_camera_orientation(phi=75 * DEGREES, theta=0* DEGREES)

        surface = Surface(
            lambda u, v: np.array([u, v, np.exp(-u**2 - v**2)]),
            u_range=[-3, 3],
            v_range=[-3, 3],
            resolution=(30, 30),
            fill_opacity=0.8,
            checkerboard_colors=[BLUE_D, BLUE_E]
        ).move_to(DOWN*2)

        sphere = Sphere(radius=1, fill_opacity=0.5).set_color(GREEN).move_to(UP*5)

        tex1 = Tex(r"Gauss map $n: X \to S^2$").

        self.add(surface, sphere)
        self.wait()


Manim Community v0.18.1

In [99]:
%%manim -qh BumpyCurve

from manim import *
import numpy as np

class BumpyCurve(Scene):
    def construct(self):
        # --- Bump Curve Setup ---
        def curve_func(x):
            # Modified bump curve with additional oscillations
            return 2 * np.exp(-7 * x**2) + 0.02*np.sin(7*x) + 0.02*np.cos(6*x)
        
        # Parameters for reparameterization
        x_min, x_max = -5, 3
        num_points = 1000
        xs = np.linspace(x_min, x_max, num_points)
        ys = curve_func(xs)
        # Apply vertical shift to match the drawn curve (UP*0.7)
        points = np.stack([xs, ys + 0.7], axis=1)
        # Compute distances between consecutive points to approximate arc length
        distances = np.linalg.norm(np.diff(points, axis=0), axis=1)
        cumulative = np.concatenate(([0], np.cumsum(distances)))
        total_length = cumulative[-1]
        
        # Create the bump curve using the original parameterization (for display)
        curve = FunctionGraph(curve_func, x_range=[x_min, x_max], color=BLUE).shift(UP*0.7)
        self.play(Create(curve))
        
        # Helper: Given an arc length s, return the corresponding x value by interpolation
        def x_from_s(s):
            return np.interp(s, cumulative, xs)
        
        # ValueTracker for arc-length parameter (instead of x)
        arc_length_tracker = ValueTracker(0)
        
        # Dot that always updates its position along the curve (constant speed)
        moving_dot = always_redraw(lambda: Dot(
            point=np.array([
                x_from_s(arc_length_tracker.get_value()),
                curve_func(x_from_s(arc_length_tracker.get_value())) + 0.7,
                0
            ]),
            color=RED
        ))
        
        # Arrow on the curve that always updates to show the unit normal vector
        normal_arrow = always_redraw(lambda: self.get_normal_arrow(
            x_from_s(arc_length_tracker.get_value()), curve_func, shift=0.7
        ))
        
        self.play(FadeIn(moving_dot, normal_arrow))
        
        # --- Gauss Map Setup ---
        # Create a unit circle to represent the Gauss map.
        # Its center is fixed so that the arrow (Gauss map) always originates at the center.
        circle_center = RIGHT*4 + UP*2
        unit_circle = Circle(radius=1, color=GREEN).move_to(circle_center)
        self.play(Create(unit_circle))
        
        # Dot at the center of the unit circle (fixed)
        center_dot = Dot(point=circle_center, color=WHITE)
        
        # Gauss map arrow: from the center of the unit circle to its circumference,
        # using the unit normal vector from the bump curve.
        gauss_arrow = always_redraw(lambda: Arrow(
            circle_center,
            circle_center + self.get_unit_normal(
                x_from_s(arc_length_tracker.get_value()), curve_func
            ),
            buff=0,
            color=YELLOW
        ))
        
        # Add a text label to indicate the Gauss map.
        gauss_text = Tex(r"Gauss Map $n: X \to S^1$", font_size=50).next_to(unit_circle, UP)
        
        self.play(FadeIn(center_dot, gauss_arrow, gauss_text))
        
        # Animate the dot along the curve at constant speed
        self.play(arc_length_tracker.animate.set_value(total_length), run_time=5, rate_func=linear)
        self.wait(1)
        self.play(arc_length_tracker.animate.set_value(0), run_time=5, rate_func=linear)

        
    def get_normal_arrow(self, x, func, shift=0):
        # Compute the unit normal at a point on the bump curve for the on-curve arrow
        eps = 1e-5
        f_x = func(x)
        f_prime = (func(x + eps) - func(x - eps)) / (2 * eps)
        tangent = np.array([1, f_prime, 0])
        tangent_unit = tangent / np.linalg.norm(tangent)
        # Normal vector: rotate tangent 90° counterclockwise
        normal_unit = np.array([-tangent_unit[1], tangent_unit[0], 0])
        pos = np.array([x, f_x + shift, 0])
        return Arrow(pos, pos + normal_unit, buff=0, color=YELLOW)
    
    def get_unit_normal(self, x, func):
        # Compute and return the unit normal vector from the bump curve (for the Gauss map)
        eps = 1e-5
        f_prime = (func(x + eps) - func(x - eps)) / (2 * eps)
        tangent = np.array([1, f_prime, 0])
        tangent_unit = tangent / np.linalg.norm(tangent)
        normal_unit = np.array([-tangent_unit[1], tangent_unit[0], 0])
        return normal_unit  # Already unit-length


Manim Community v0.18.1

In [98]:
%%manim -qm BumpyPlane

from manim import *
import numpy as np

class BumpyPlane(ThreeDScene):
    def construct(self):
        self.set_camera_orientation(phi=75 * DEGREES, theta=0 * DEGREES)
        
        # Surface definition: F(u,v) = (u, v, exp(-u^2 - v^2))
        def surface_func(u, v):
            return np.array([u, v, np.exp(-u**2 - v**2)])
        
        # Create the surface and shift it down by 2 units
        surface = Surface(
            lambda u, v: surface_func(u, v),
            u_range=[-3, 3],
            v_range=[-3, 3],
            resolution=(30, 30),
            fill_opacity=0.8,
            checkerboard_colors=[BLUE_D, BLUE_E]
        ).shift(np.array([0, -2, 0]))
        
        # Create the sphere for the Gauss map (center fixed at UP*5)
        sphere_center = np.array([0, 5, 0])
        sphere = Sphere(radius=1, fill_opacity=0.5).set_color(GREEN).move_to(sphere_center)
        
        # Add a red dot on the top of the surface
        top_point = surface_func(0, 0) + np.array([0, -2, 0])  # Adjust for surface position
        red_dot = Dot3D(point=top_point, color=RED, radius=0.05)
        
        # Compute normal at red dot position (gradient of the surface function)
        normal_top = np.array([0, 0, 1])  # Approximate normal at (0,0)
        normal_arrow_top = Arrow3D(start=top_point, end=top_point + 0.8 * normal_top, color=YELLOW)
        
        # Add a white dot at the center of the sphere
        white_dot = Dot3D(point=sphere_center, color=WHITE, radius=0.05)
        
        # Add a yellow normal vector at the white dot (same direction as on surface)
        normal_arrow_sphere = Arrow3D(start=sphere_center, end=sphere_center + normal_top, color=YELLOW)
        
        # Add fixed text for the Gauss map
        tex1 = Tex(r"Gauss map $n: X \to S^2$", font_size=50).move_to(UP*1.5 + RIGHT*4)
        self.add_fixed_in_frame_mobjects(tex1)
        
        self.add(surface, sphere, red_dot, normal_arrow_top, white_dot, normal_arrow_sphere)
        self.wait()



Manim Community v0.18.1

In [100]:
%%manim -qh BumpyPlane

from manim import *
import numpy as np

class BumpyPlane(ThreeDScene):
    def construct(self):
        self.set_camera_orientation(phi=75 * DEGREES, theta=0 * DEGREES)
        
        # Surface definition: F(u,v) = (u, v, exp(-u^2 - v^2))
        def surface_func(u, v):
            return np.array([u, v, np.exp(-u**2 - v**2)])
        
        # Create the surface and shift it down by 2 units
        surface = Surface(
            lambda u, v: surface_func(u, v),
            u_range=[-3, 3],
            v_range=[-3, 3],
            resolution=(30, 30),
            fill_opacity=0.8,
            checkerboard_colors=[BLUE_D, BLUE_E]
        ).shift(np.array([0, -2, 0]))
        
        # Create the sphere for the Gauss map (center fixed at UP*5)
        sphere_center = np.array([0, 5, 0])
        sphere = Sphere(radius=1, fill_opacity=0.5).set_color(GREEN).move_to(sphere_center)
        
        # Add a red dot on the top of the surface
        top_point = surface_func(0, 0) + np.array([0, -2, 0])  # Adjust for surface position
        red_dot = Dot3D(point=top_point, color=RED, radius=0.05)
        
        # Compute normal at red dot position (gradient of the surface function)
        normal_top = np.array([0, 0, 1])  # Approximate normal at (0,0)
        normal_arrow_top = Arrow3D(start=top_point, end=top_point + 0.8 * normal_top, color=YELLOW)
        
        # Add a white dot at the center of the sphere
        white_dot = Dot3D(point=sphere_center, color=WHITE, radius=0.05)
        
        # Add a yellow normal vector at the white dot (same direction as on surface)
        normal_arrow_sphere = Arrow3D(start=sphere_center, end=sphere_center + normal_top, color=YELLOW)
        
        # Create a tangent plane on top of the surface at the red dot
        tangent_plane_surface = Surface(
            lambda u, v: np.array([u, v, top_point[2]]),
            u_range=[-0.8, 0.8],
            v_range=[-0.8, 0.8],
            resolution=1,
            fill_opacity=0.6,
            fill_color=YELLOW,
            stroke_color=YELLOW,
            checkerboard_colors=[YELLOW, YELLOW]
        ).move_to(top_point)
        tangent_plane_surface.z_index=1
        
        # Create a tangent plane on top of the sphere at the white dot
        tangent_plane_sphere = Surface(
            lambda u, v: np.array([u, v, sphere_center[2] + 1]),
            u_range=[-0.8, 0.8],
            v_range=[-0.8, 0.8],
            resolution=1,
            fill_opacity=0.6,
            fill_color=YELLOW,
            stroke_color=YELLOW,
            checkerboard_colors=[YELLOW, YELLOW]
        ).move_to(sphere_center + normal_top)
        tangent_plane_sphere.z_index=1
        
        # Add fixed text for the Gauss map
        tex1 = Tex(r"Gauss map $n: X \to S^2$", font_size=50).move_to(UP*1.5 + RIGHT*4)
        self.add_fixed_in_frame_mobjects(tex1)
        self.remove(tex1)

        def curve_func(t):
            return surface_func(t, t)

        tracker = ValueTracker(-2)

        red_dot = always_redraw(lambda: Dot3D(point=curve_func(tracker.get_value())+2*DOWN, color=RED, radius=0.05))


        #normal_arrow_sphere = Arrow3D(start=sphere_center, end=sphere_center + normal_in_plane_vector(-2), color=YELLOW)
        normal_arrow_sphere = always_redraw(lambda: Arrow3D(start=sphere_center, end=sphere_center + normal_in_plane_vector(tracker.get_value()), color=YELLOW))
        #normal_arrow_top = Arrow3D(start=curve_func(-2)+2*DOWN, end=curve_func(-2)+2*DOWN + 0.8 * normal_in_plane_vector(-2), color=YELLOW)
        normal_arrow_top = always_redraw(lambda: Arrow3D(start=curve_func(tracker.get_value())+2*DOWN, end=curve_func(tracker.get_value())+2*DOWN + 0.8 * normal_in_plane_vector(tracker.get_value()), color=YELLOW))

        
        #self.add(surface, sphere, red_dot, normal_arrow_top, white_dot, normal_arrow_sphere)
        self.play(Create(surface))
        self.play(FadeIn(red_dot, normal_arrow_top))
        self.play(Create(sphere))
        self.play(FadeIn(white_dot, normal_arrow_sphere, tex1))
                 #tangent_plane_surface, tangent_plane_sphere)
        self.play(tracker.animate.set_value(2), rate_func=linear,run_time = 5)
        self.wait(1)
        self.play(tracker.animate.set_value(-2), rate_func=linear,run_time = 5)


Manim Community v0.18.1

In [3]:
%%manim -qh ShapeOperatorNew

from manim import *
import numpy as np

def normal_in_plane_vector(t):
    # Define the curve r(t) = [t, t, exp(-2t^2)]
    r_t = np.array([t, t, np.exp(-2 * t**2)])
    
    # Compute the tangent vector dr/dt (the derivative of r(t))
    dr_dt = np.array([1, 1, -4 * t * np.exp(-2 * t**2)])

    normal_3d = np.array([-dr_dt[2], -dr_dt[2], dr_dt[0]])
    
    # Normalize the normal vector to have unit length
    normal_unit = normal_3d / np.linalg.norm(normal_3d)
    
    return normal_unit

class ShapeOperatorNew(ThreeDScene):
    def construct(self):
        self.set_camera_orientation(phi=75 * DEGREES, theta=0 * DEGREES)
        
        # Surface definition: F(u,v) = (u, v, exp(-u^2 - v^2))
        def surface_func(u, v):
            return np.array([u, v, np.exp(-u**2 - v**2)])
        
        # Create the surface and shift it to the center (0,0,0)
        surface = Surface(
            lambda u, v: surface_func(u, v),
            u_range=[-3, 3],
            v_range=[-3, 3],
            resolution=(30, 30),
            fill_opacity=0.8,
            checkerboard_colors=[BLUE_D, BLUE_E]
        )
        
        # Red dot at the center of the surface (at u=0, v=0)
        red_dot = Dot3D(point=surface_func(0, 0), color=RED, radius=0.05)
        red_dot.z_index=1
        
        # Define a yellow curve passing through the red dot
        def curve_func(t):
            return surface_func(t, t)  # v=0, just a curve in the u direction
        
        curve = ParametricFunction(
            curve_func,
            t_range=np.array([-2, 2]),
            color=YELLOW
        )
        
        # Compute the tangent vector to the curve at the red dot
        curve_tangent = np.array([1, 1, 0])  # Tangent at (0, 0)
        tangent_vector = Arrow3D(
            start=surface_func(0, 0), 
            end=surface_func(0, 0) + 0.75*curve_tangent, 
            color=GREEN
        )
        
        # Add fixed text
        tex1 = MathTex(r"\gamma:[-1, 1] \to X, \gamma(0)=p", font_size=70).to_edge(UP)
        tex2 = Tex(r"$v=\gamma'(0) \in T_pX$, $T_pX$ tangent space at $p$").next_to(tex1, DOWN)
        self.add_fixed_in_frame_mobjects(tex1, tex2)
        self.remove(tex1, tex2)
        
        # Add surface, red dot, curve, and tangent vector
        self.add(surface, red_dot)
        self.play(FadeIn(tex1))
        self.play(Create(curve))
        self.wait()
        self.play(Create(tangent_vector), FadeIn(tex2))
        self.wait()

        tangent_plane_surface = Surface(
            lambda u, v: np.array([u, v, 1]),
            u_range=[-0.8, 0.8],
            v_range=[-0.8, 0.8],
            resolution=1,
            fill_opacity=0.6,
            fill_color=GREEN,
            stroke_color=GREEN,
            checkerboard_colors=[GREEN, GREEN]
        ).move_to(red_dot)
        tangent_plane_surface.z_index=1
        self.play(FadeIn(tangent_plane_surface))
        self.wait()


        sphere_center = np.array([0, 5, 0])
        sphere = Sphere(radius=1, fill_opacity=0.5).set_color(GREEN).move_to(sphere_center)
        white_dot = Dot3D(point=sphere_center, color=WHITE, radius=0.05)
        normal_top = np.array([0, 0, 1])
        
        tangent_plane_sphere = Surface(
            lambda u, v: np.array([u, v, sphere_center[2] + 1]),
            u_range=[-0.8, 0.8],
            v_range=[-0.8, 0.8],
            resolution=1,
            fill_opacity=0.6,
            fill_color=YELLOW,
            stroke_color=YELLOW,
            checkerboard_colors=[YELLOW, YELLOW]
        ).move_to(sphere_center + normal_top)
        tangent_plane_sphere.z_index=1

        tracker = ValueTracker(-2)

        #normal_arrow_sphere = Arrow3D(start=sphere_center, end=sphere_center + normal_in_plane_vector(-2), color=YELLOW)
        normal_arrow_sphere = always_redraw(lambda: Arrow3D(start=sphere_center, end=sphere_center + normal_in_plane_vector(tracker.get_value()), color=YELLOW))
        #normal_arrow_top = Arrow3D(start=curve_func(-2)+2*DOWN, end=curve_func(-2)+2*DOWN + 0.8 * normal_in_plane_vector(-2), color=YELLOW)
        normal_arrow_top = always_redraw(lambda: Arrow3D(start=curve_func(tracker.get_value())+2*DOWN, end=curve_func(tracker.get_value())+2*DOWN + 0.8 * normal_in_plane_vector(tracker.get_value()), color=YELLOW))


        plot1 = VGroup(surface, red_dot, curve, tangent_vector, tangent_plane_surface)
        plot2 = VGroup(sphere, white_dot, normal_arrow_sphere, tangent_plane_sphere)
        
        self.play(plot1.animate.shift((0,-2,0)))
        self.play(FadeIn(plot2[0:2]))
        self.wait()
        self.play(FadeIn(normal_arrow_sphere, normal_arrow_top))
        self.play(tracker.animate.set_value(2), rate_func=linear,run_time = 2)
        self.play(tracker.animate.set_value(0), rate_func=linear, run_time=1)
        self.wait()

        tanvec2 = Arrow3D(start=sphere_center + normal_in_plane_vector(0), end=sphere_center + normal_in_plane_vector(0) + 0.75*np.array([1,1,0])).set_color(BLUE)

        gauss = MathTex(r"\nabla_v n(p) = (n \circ \gamma)'(0) \in T_{n(p)}S^2").move_to(DOWN*1.5)
        diff1 = MathTex(r"dn_p: T_pX \to T_{n(p)}S^2, v \mapsto \nabla_v n(p)").next_to(gauss, DOWN)
        shape = Tex(r"Shape operator $S_p: T_pX \to T_pX, v \mapsto -dn_p(v)$").next_to(diff1, DOWN)
        self.add_fixed_in_frame_mobjects(gauss, diff1, shape)
        self.remove(gauss, diff1, shape)

        self.play(FadeIn(gauss), FadeIn(tangent_plane_sphere), FadeIn(tanvec2))
        self.wait()
        self.play(Write(diff1))
        self.wait()
        self.play(Write(shape))
        self.wait()

        

Manim Community v0.18.1